In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Tb₂Ti₂O₇ on HEiDi

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the pyrochlore Tb₂Ti₂O₇ to constant-wavelength neutron single-crystal data from HEiDi at
MLZ, with anisotropic displacements and Becker-Coppens extinction.

The project is the catalogue case `sc-neut-cwl_tb2ti2o7-heidi`, the one the command line and the
app's Examples open.

In [ ]:
import easydiffraction as edi

## Load the project

In [ ]:
project_path = edi.download_data('sc-neut-cwl_tb2ti2o7-heidi', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['tb2ti2o7']
experiment = project.experiments['heidi']

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [ ]:
for parameter in project.free_parameters:
    parameter.free = False

In [ ]:
project.display.pattern('heidi')

## Perform the analysis

One new group per fit: the scale, O1's position, the displacement tensors, then the extinction.
diffraction-lib also fits occupancies and isotropic displacements, which this case keeps fixed.

### Scale

The scale brings the calculated intensities to the measured ones.

In [ ]:
experiment.linked_structures['tb2ti2o7'].scale.free = True

In [ ]:
project.display.parameters.free()

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('heidi')

### O1's position

The one coordinate the symmetry leaves free.

In [ ]:
structure.atom_sites['O1'].fract_x.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('heidi')

### Anisotropic displacements

In [ ]:
for tensor in structure.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('heidi')

### Extinction

The domain radius and the mosaic spread.

In [ ]:
experiment.extinction.domain_radius.free = True
experiment.extinction.mosaic_spread.free = True

In [ ]:
project.analysis.fit()
project.display.fit.results()

In [ ]:
project.display.pattern('heidi')

In [ ]:
project.display.fit.correlations()

In [ ]:
project.display.structure('tb2ti2o7')

## Save the project

In [ ]:
project.save_as('sc-neut-cwl_tb2ti2o7-heidi')
project.report.save_html('sc-neut-cwl_tb2ti2o7-heidi.html')